# Test `Fast_VTON_full.pt` trên Kaggle (P100) / Colab Pro

Notebook test model virtual try-on `Fast_VTON_full.pt` (Stage 1) đã huấn luyện xong. Chạy được cả **Kaggle (P100)** và **Google Colab Pro (A100/V100)**.

**Luồng:** cài dependencies (Fast-VTON + dự án test) → lấy file model vào `models/` → kiểm tra nhanh pipeline → chạy Gradio để up ảnh người + ảnh quần áo.

> **Quan trọng:** bật **Internet** trong Settings (Kaggle) để tải Segformer (auto-agnostic) và scheduler. Trên Kaggle, Gradio dùng proxy có sẵn (không cần `share`); trên Colab dùng `share=True`.
>
> **P100 lưu ý:** không có tensor cores / bf16, nhưng bundle là **fp16** nên chạy được. Mỗi ảnh ~1.5–3s. Phải `pip uninstall -y peft` (xung đột với diffusers 0.22).

In [ ]:
import os
import sys

IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/input') or ('KAGGLE_CONTAINER_NAME' in os.environ)
RUNTIME = 'Colab' if IS_COLAB else ('Kaggle' if IS_KAGGLE else 'local')
print('Runtime:', RUNTIME)

WORKDIR = os.getcwd()
PARENT = os.path.dirname(WORKDIR)
FAST_VTON_DIR = os.path.join(PARENT, 'Fast-VTON')
print('WORKDIR   :', WORKDIR)
print('Fast-VTON :', FAST_VTON_DIR)

if not os.path.isdir(FAST_VTON_DIR):
    !git clone -q https://github.com/hoangtung386/Fast-VTON.git {FAST_VTON_DIR}
else:
    print('Fast-VTON đã có sẵn')

## Bước 1 — Cài đặt môi trường

Pin đúng version theo Fast-VTON (`diffusers==0.22.0`, `transformers==4.37.2`, `torch==2.2.1`), cài Fast-VTON, rồi cài dự án test (`fast_vton_test`). Cuối cùng **gỡ `peft`** để diffusers không bật PEFT backend gây vỡ giữa lượt UNet.

In [ ]:
!pip install -q torch==2.2.1 torchvision==0.17.1
!pip install -q -e {FAST_VTON_DIR}[vton]
!pip install -q numpy==1.26.4
!pip uninstall -q -y peft
!pip install -q -e {WORKDIR}

## Bước 2 — Lấy file model `Fast_VTON_full.pt`

Sửa `MODEL_SOURCE` và các đường dẫn tương ứng:

- `'drive'` — Colab: mount Drive rồi copy từ `GDRIVE_MODEL_PATH`.
- `'gdown'` — tải từ Google Drive bằng file id (`GDRIVE_FILE_ID`).
- `'kaggle'` — copy từ Kaggle Dataset input (`KAGGLE_MODEL_PATH`, ví dụ `/kaggle/input/<dataset>/Fast_VTON_full.pt`).

File sẽ được đặt vào `models/Fast_VTON_full.pt` (đúng `DEFAULT_BUNDLE`).

In [ ]:
import os
import shutil

MODEL_DIR = os.path.join(WORKDIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, 'Fast_VTON_full.pt')

MODEL_SOURCE = 'drive'
GDRIVE_MODEL_PATH = '/content/drive/MyDrive/path/to/Fast_VTON_full.pt'
GDRIVE_FILE_ID = 'YOUR_FILE_ID'
KAGGLE_MODEL_PATH = '/kaggle/input/your-dataset/Fast_VTON_full.pt'

if not os.path.exists(MODEL_PATH):
    if MODEL_SOURCE == 'kaggle':
        shutil.copy(KAGGLE_MODEL_PATH, MODEL_PATH)
    elif MODEL_SOURCE == 'gdown':
        !pip install -q gdown
        !gdown {GDRIVE_FILE_ID} -O {MODEL_PATH}
    elif MODEL_SOURCE == 'drive':
        if IS_COLAB:
            from google.colab import drive
            drive.mount('/content/drive')
        shutil.copy(GDRIVE_MODEL_PATH, MODEL_PATH)
    print('đã chuẩn bị', MODEL_PATH)
else:
    print('model đã có sẵn tại', MODEL_PATH)

## Bước 3 — Kiểm tra nhanh pipeline (bắt lỗi môi trường sớm)

Chạy thử trên ảnh synthetic để xác nhận: bundle load được, DINOv2/CLIP/VAE/inversion hoạt động, mask + forward chạy qua. Ảnh rác nên kết quả chỉ để verify không crash.

In [ ]:
import os
import torch
from PIL import Image
from fast_vton_test.inference import FastVTONInference

bundle = os.path.join(WORKDIR, 'models', 'Fast_VTON_full.pt')
pred = FastVTONInference(bundle, device='cuda')
person = Image.new('RGB', (384, 512), (220, 200, 180))
garment = Image.new('RGB', (224, 224), (30, 90, 200))
agnostic = pred.build_agnostic(person)
out = pred.try_on(person, agnostic, garment)
print('bundle step :', pred.bundle.manifest.step)
print('output shape:', tuple(out.shape))

## Bước 4 — Chạy Gradio để test thật

Mở link hiện ra, up **ảnh người mẫu** + **ảnh quần áo**, bấm *Thử đồ*. Ảnh agnostic tự sinh (human parsing). Để tắt auto-agnostic, bỏ tick và up sẵn ảnh agnostic.

In [ ]:
%cd {WORKDIR}
from fast_vton_test.app import build_demo
build_demo().launch(share=IS_COLAB, debug=False)

## (Tuỳ chọn) Chạy qua CLI thay vì Gradio

Đặt ảnh test vào `data/` rồi chạy. Hữu ích khi chạy batch hoặc không cần UI.

In [ ]:
print('CLI test:')
print('  python -m fast_vton_test.cli --person data/nguoi.jpg --garment data/ao.jpg --auto-agnostic --output outputs/result.png')
print('Tắt auto-agnostic nếu có sẵn ảnh agnostic:')
print('  python -m fast_vton_test.cli --person data/nguoi.jpg --garment data/ao.jpg --agnostic data/agnostic.jpg --output outputs/result.png')